#  Fine-Tuning Nomic Embed Text MoE v2 trên Google Colab GPU T4

Notebook này cho phép bạn huấn luyện mô hình **Nomic MoE v2** (`nomic-ai/nomic-embed-text-v2-moe`) trên bộ dữ liệu 2.158 câu hỏi UTH thực tế với GPU T4 miễn phí trên Google Colab.

###  Hướng dẫn sử dụng:
1. Đảm bảo bạn đã chọn **T4 GPU**: Vào `Runtime` -> `Change runtime type` -> chọn `T4 GPU`.
2. Upload thư mục `dataset/` (chứa `final_train_set.jsonl` & `final_val_set.jsonl`) lên Colab.
3. Chạy từng Cell theo thứ tự bên dưới.
4. Sau khi train xong, file `uth-nomic-embed-v2.zip` sẽ được tự động nén và tải về máy.

In [ ]:
# 1. Kiểm tra phần cứng GPU T4
!nvidia-smi

In [ ]:
# 2. Cài đặt các thư viện phụ thuộc
!pip install -q sentence-transformers>=3.0.0 transformers>=4.40.0 peft>=0.10.0 datasets accelerate einops pydantic scikit-learn

In [ ]:
# 3. Kiểm tra hoặc Upload bộ dữ liệu (final_train_set.jsonl & final_val_set.jsonl)
import os
from google.colab import files

os.makedirs("dataset", exist_ok=True)
train_exist = os.path.exists("dataset/final_train_set.jsonl") or os.path.exists("final_train_set.jsonl")
val_exist = os.path.exists("dataset/final_val_set.jsonl") or os.path.exists("final_val_set.jsonl")

if not (train_exist and val_exist):
    print("⚠️ Chưa tìm thấy đủ file dữ liệu! Hãy upload 2 file final_train_set.jsonl và final_val_set.jsonl:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".jsonl"):
            os.rename(fname, f"dataset/{fname}")
    print("✅ Upload dữ liệu hoàn tất!")
else:
    print("✅ Bộ dữ liệu final_train_set.jsonl và final_val_set.jsonl đã sẵn sàng!")

In [ ]:
# 4. Chạy quá trình Fine-Tuning Nomic MoE v2 với GPU T4 (fp16, batch_size=32)
!python 02_train_nomic_colab.py

In [ ]:
# 5. Đánh giá & So sánh chỉ số Benchmark (MRR@10 & NDCG@10) giữa mô hình gốc và mô hình vừa Fine-tuned
!python 03_evaluate_model.py

In [ ]:
# 6. Tải file weights mô hình uth-nomic-embed-v2.zip về máy cục bộ
from google.colab import files
if os.path.exists("uth-nomic-embed-v2.zip"):
    files.download("uth-nomic-embed-v2.zip")
elif os.path.exists("fine-tune-nomic/uth-nomic-embed-v2.zip"):
    files.download("fine-tune-nomic/uth-nomic-embed-v2.zip")
else:
    print("Chưa tìm thấy file uth-nomic-embed-v2.zip.")